# Scenario Tear Sheet

**Use after source analytics:** Reporting notebooks render results produced by the pricing, analytics, statement, and portfolio notebooks; they do not replace those calculation workflows.

**Purpose:** Summarize scenario impacts, driver sensitivities, fan charts, and variance decomposition for stress-review workflows.

**Prerequisites:** `05_portfolio_and_scenarios/scenarios_and_stress_testing.ipynb`.

**What you'll learn:**

- Prepare scenario comparison data.
- Render `reporting.scenario_tearsheet` for stress-test review.
- Separate scenario calculation from presentation.

Stress a forecast four ways — a driver tornado, a base/upside/downside
comparison, a Monte-Carlo percentile fan, and a variance table — and render them
with `scenario_tearsheet`. Each input is produced by a real engine call.


In [1]:
import datetime as dt
import json

from finstack_quant import reporting
from finstack_quant.statements import Evaluator, ForecastSpec, ModelBuilder
from finstack_quant.statements_analytics import (
    evaluate_scenario_set,
    generate_tornado_entries,
    run_monte_carlo,
    run_sensitivity,
    run_variance,
)


def build(model_id, revenue, cogs, opex, *, stochastic_revenue=False):
    qs = ["2025Q1", "2025Q2", "2025Q3", "2025Q4"]
    b = ModelBuilder(model_id)
    b.periods("2025Q1..Q4", "2025Q2")
    if stochastic_revenue:
        # Actuals only; forecast periods are sampled stochastically for the MC fan.
        b.value("revenue", [("2025Q1", revenue[0]), ("2025Q2", revenue[1])])
        b.forecast("revenue", ForecastSpec.normal(mean=0.05, std_dev=0.15, seed=42))
    else:
        b.value("revenue", list(zip(qs, revenue)))
    b.value("cogs", list(zip(qs, cogs)))
    b.compute("gross_profit", "revenue - cogs")
    b.value("opex", list(zip(qs, opex)))
    b.compute("ebitda", "gross_profit - opex")
    return b.build()


base = build("scenario-base", [100, 110, 115, 120], [60, 65, 68, 72], [20, 22, 23, 24])
result = Evaluator().evaluate(base)
model_json = base.to_json()
print("Base EBITDA 2025Q4:", result.get("ebitda", "2025Q4"))

Base EBITDA 2025Q4: 24.0


## Driver tornado

`run_sensitivity` perturbs a forecast driver; `generate_tornado_entries` ranks
the impact on the target metric.

In [2]:
sens_cfg = json.dumps({
    "mode": "Diagonal",
    "parameters": [{
        "node_id": "revenue",
        "period_id": "2025Q3",
        "base_value": 115.0,
        "perturbations": [95.0, 105.0, 115.0, 125.0, 135.0],
    }],
    "target_metrics": ["ebitda"],
})
tornado = json.loads(generate_tornado_entries(run_sensitivity(model_json, sens_cfg), "ebitda", "2025Q3"))
print("Tornado:", tornado)

Tornado: [{'parameter_id': 'revenue', 'downside': -20.0, 'upside': 20.0}]


## Scenario comparison

`evaluate_scenario_set` applies named overrides to the forecast periods; we read
the target metric from each scenario.

In [3]:
scen_cfg = json.dumps({"scenarios": {
    "upside": {"overrides": {"revenue": 135.0}},
    "downside": {"overrides": {"revenue": 100.0}},
}})
scen_out = json.loads(evaluate_scenario_set(model_json, scen_cfg))
scenarios = {"base": result.get("ebitda", "2025Q4")}
scenarios.update({name: sr["nodes"]["ebitda"]["2025Q4"] for name, sr in scen_out.items()})
print("Scenarios (EBITDA 2025Q4):", scenarios)

Scenarios (EBITDA 2025Q4): {'base': 24.0, 'upside': 39.0, 'downside': 4.0}


## Monte-Carlo fan

A stochastic revenue forecast (`ForecastSpec.normal`) drives the simulation; we
read the 5th / 50th / 95th percentiles per forecast period into a fan. The band
widens with the driver's volatility.

In [4]:
mc_model = build("scenario-mc", [100, 110, 115, 120], [60, 65, 68, 72], [20, 22, 23, 24], stochastic_revenue=True).to_json()
mc = json.loads(run_monte_carlo(mc_model, json.dumps({"n_paths": 2000, "seed": 42, "percentiles": [0.05, 0.5, 0.95]})))
periods = mc["forecast_periods"]
vals = mc["percentile_results"]["ebitda"]["values"]
monte_carlo = {
    "periods": periods,
    "p_low": [vals[p][0][1] for p in periods],
    "p_mid": [vals[p][1][1] for p in periods],
    "p_high": [vals[p][2][1] for p in periods],
}
print("MC fan:", monte_carlo)

MC fan: {'periods': ['2025Q3', '2025Q4'], 'p_low': [18.811931278092665, 13.764966166126467], 'p_mid': [19.04618672700431, 14.09855233285213], 'p_high': [19.28741483667644, 14.455966621677529]}


## Variance vs an alternate plan

In [5]:
plan = build("scenario-plan", [105, 116, 124, 132], [62, 67, 70, 74], [21, 23, 24, 25])
plan_eval = Evaluator().evaluate(plan)
variance = json.loads(run_variance(
    result.to_json(),
    plan_eval.to_json(),
    json.dumps({"baseline_label": "base", "comparison_label": "plan", "metrics": ["revenue", "ebitda"], "periods": ["2025Q3", "2025Q4"]}),
))

## Forward-risk tear sheet

In [6]:
reporting.scenario_tearsheet(
    tornado=tornado,
    scenarios=scenarios,
    monte_carlo=monte_carlo,
    variance=variance,
    target_metric="ebitda",
    title="Acme Corp — Forward Risk",
    generated=dt.date(2026, 6, 22),
)

Period,Metric,Baseline,Comparison,Abs Δ,% Δ
2025Q3,revenue,115.00,124.00,9.00,+7.8%
2025Q4,revenue,120.00,132.00,12.00,+10.0%
2025Q3,ebitda,24.00,30.00,6.00,+25.0%
2025Q4,ebitda,24.00,33.00,9.00,+37.5%


## Saving a standalone HTML file

```python
ts = reporting.scenario_tearsheet(tornado=tornado, scenarios=scenarios, monte_carlo=monte_carlo, variance=variance, generated=dt.date(2026, 6, 22))
ts.save("scenario_tearsheet.html")
```

## Takeaways

- Reporting functions are presentation wrappers over analytics, valuation, statement, or portfolio results produced earlier in the curriculum.
- Keep the analytical source of truth in the typed objects or JSON specs, then render a tear sheet for review.
- Pass fixed `generated` dates in examples so notebook output remains reproducible.
